# Exploration des données traitées (Processed Data)

Ce notebook a pour but d'explorer les fichiers générés après les étapes de nettoyage (clean), d'ingénierie des caractéristiques (feature engineering) et de construction de la cible (target building). Nous allons visualiser les colonnes et quelques échantillons des fichiers suivants :
1. `main_table.csv`
2. `feature_matrix.csv`
3. `training_set.csv`

## 1. Main Table (`main_table.csv`)

Il s'agit de la table principale fusionnant les lignes de commandes, les informations sur les factures et les données de localisation (GPS) des clients. C'est la donnée de base, propre, avant toute agrégation complexe ou calcul de features ML.

In [1]:
import pandas as pd

main_table_path = "../data/processed/main_table.csv"
df_main = pd.read_csv(main_table_path, parse_dates=["date_commande"])

print(f"Shape: {df_main.shape}")
print(f"Colonnes: {list(df_main.columns)}\n")
display(df_main.sample(5))

Shape: (78530, 18)
Colonnes: ['code_facture', 'quantite', 'code_article', 'designation', 'categorie', 'societe_ligne', 'is_bulk_order', 'code_client', 'date_commande', 'ref_commercial', 'societe', 'mois', 'annee', 'annee_mois', 'jour_semaine', 'trimestre', 'latitude', 'longitude']



,code_facture,quantite,code_article,designation,categorie,societe_ligne,is_bulk_order,code_client,date_commande,ref_commercial,societe,mois,annee,annee_mois,jour_semaine,trimestre,latitude,longitude
52198,V25-FAC+03533,2,2409BRN2CABLUE6/128,REDMI 14C STARRY BLUE 6/128GB,GSM XIAOMI,LSAT,False,CLT109461,2025-06-17,MH,LSAT,6,2025,2025-06,1,2,35.640109,10.967749
17694,V24-FAC+04789,31,M2236E1 WHITE,REDMI BUDS 4 LITE WHITE,ACC XIAOMI,LSAT,False,CLT012878,2024-07-15,IBD,LSAT,7,2024,2024-07,0,3,NaN,NaN
21995,V24-FAC+05779,24,286838745,NOKIA 150 DS NENA2 BLACK,GSM NOKIA,LSAT,False,CLT107878,2024-08-21,MH,LSAT,8,2024,2024-08,2,3,35.627137,10.761428
15516,V24-FAC+04267,2,BG6 MAGIC SKIN GREEN,TECNO SPARK GO 2024 MAGIC SKIN GREEN 4+128,GSM TECNO,LSAT,False,CLT112568,2024-06-19,IS,LSAT,6,2024,2024-06,2,2,NaN,NaN
55566,V25-FAC+04385,100,M2420E1 BLACK,REDMI BUDS 6 PLAY BLACK,ACC XIAOMI,LSAT,False,CLT106366,2025-07-18,FAB,LSAT,7,2025,2025-07,4,3,34.746024,10.760302


## 2. Feature Matrix (`feature_matrix.csv`)

Ce fichier contient les caractéristiques (features) construites pour le Machine Learning. Le script de Feature Engineering a agrégé les historiques d'achats pour chaque paire (Client, Produit) afin de créer des indicateurs prédictifs divisés en 5 groupes :
- **Historique de commandes** (fréquence, quantité moyenne, récence, tendance...)
- **Saisonnalité** (meilleur mois, coefficient saisonnier...)
- **Géographie** (latitude, longitude, présence de GPS...)
- **Produit** (catégorie, nouveauté, produit en vrac...)
- **Profil Client** (nombre total de factures, de produits achetés, panier moyen...)

In [2]:
feature_matrix_path = "../data/processed/feature_matrix.csv"
df_features = pd.read_csv(feature_matrix_path)

print(f"Shape: {df_features.shape}")
print(f"Colonnes: {list(df_features.columns)}\n")
display(df_features.sample(5))

Shape: (40765, 29)
Colonnes: ['code_client', 'code_article', 'avg_qty', 'std_qty', 'min_qty', 'max_qty', 'total_qty', 'frequency', 'last_qty', 'recency_days', 'avg_delay_days', 'recency_relative', 'trend', 'company_encoded', 'current_month_coef', 'avg_seasonal_coef', 'best_month', 'has_gps', 'latitude', 'longitude', 'categorie', 'designation', 'is_bulk_product', 'nb_clients', 'days_since_first_order', 'is_new_product', 'client_total_products', 'client_total_invoices', 'client_avg_basket_size']



,code_client,code_article,avg_qty,std_qty,min_qty,max_qty,total_qty,frequency,last_qty,recency_days,...,longitude,categorie,designation,is_bulk_product,nb_clients,days_since_first_order,is_new_product,client_total_products,client_total_invoices,client_avg_basket_size
28661,CLT110314,M2420E1 BLACK,2.0,0.0,2,2,8,4,2,150,...,NaN,ACC XIAOMI,REDMI BUDS 6 PLAY BLACK,False,348,622,False,110,52,3.807692
28112,CLT109992,2404ARN45A PINK8/128,2.0,0.0,2,2,2,1,2,664,...,NaN,GSM XIAOMI,REDMI 13 PEARL PINK 8/128,False,135,691,False,16,5,3.200000
22132,CLT107792,24117RN76G GOLD8/256,1.0,0.0,1,1,1,1,1,327,...,10.609626,GSM XIAOMI,REDMI NOTE 14 SAND GOLD 8/256GB,False,88,333,False,60,46,4.021739
13603,CLT103147,23090RA98GPURP8/256,1.0,0.0,1,1,2,2,1,637,...,10.820545,GSM XIAOMI,REDMI NOTE 13 PRO+ 5G AURORA PURPLE 8/256GB,False,69,881,False,122,33,5.757576
28499,CLT110235,23129RAA4GBLUE8/128,2.0,0.0,2,2,2,1,2,879,...,NaN,GSM XIAOMI,REDMI NOTE 13 ICE BLUE 8/128GB,False,205,881,False,92,65,3.723077


## 3. Training Set (`training_set.csv`) & Explication du "Negative Sampling"

Le **Training Set** est le jeu de données final fourni au modèle (ex: XGBoost). Il combine la `feature_matrix` avec la **cible (target)** à prédire (`target_bought` : le client a-t-il acheté ce produit le mois suivant ?).

### Qu'est-ce que le Negative Sampling (Échantillonnage Négatif) ?
Dans la vraie vie, un client n'achète qu'une infime fraction du catalogue disponible. Si l'on créait une ligne pour **toutes** les combinaisons possibles de (Client, Produit), on obtiendrait un dataset gigantesque rempli presque exclusivement de `target_bought = 0` (non acheté). Le modèle serait déséquilibré, la mémoire de la machine exploserait, et le modèle mettrait trop de temps à s'entraîner sur des cas évidents.

**La solution (implémentée dans `target_builder.py`) :**
1. **Pairs Positifs (Positive pairs) :** On garde toutes les fois où un client a effectivement commandé un produit. L'ancre est fixée au dernier mois d'achat, et on regarde le mois d'après pour voir s'il y a eu réachat (`target_bought = 1` ou `0`).
2. **Pairs Négatifs (Negative Sampling) :** Pour chaque client, on choisit aléatoirement un petit nombre de produits qu'il n'a **jamais** commandés. Dans notre cas, le ratio défini est de `3:1` (3 exemples négatifs pour 1 positif). Pour ces paires artificielles, la cible est forcément `target_bought = 0`.

Cela permet d'apprendre au modèle à quoi ressemble un "non-achat" de manière très efficace, sans exploser la taille de nos données !

In [3]:
training_set_path = "../data/processed/training_set.csv"
df_train = pd.read_csv(training_set_path)

print(f"Shape: {df_train.shape}")
print(f"Colonnes: {list(df_train.columns)}\n")

print("Distribution de la variable cible (target_bought):")
display(df_train["target_bought"].value_counts(normalize=True).apply(lambda x: f"{x*100:.1f}%"))

print("\nExemples de lignes:")
display(df_train.sample(5))

Shape: (153320, 31)
Colonnes: ['code_client', 'code_article', 'avg_qty', 'std_qty', 'min_qty', 'max_qty', 'total_qty', 'frequency', 'last_qty', 'recency_days', 'avg_delay_days', 'recency_relative', 'trend', 'company_encoded', 'current_month_coef', 'avg_seasonal_coef', 'best_month', 'has_gps', 'latitude', 'longitude', 'categorie', 'designation', 'is_bulk_product', 'nb_clients', 'days_since_first_order', 'is_new_product', 'client_total_products', 'client_total_invoices', 'client_avg_basket_size', 'target_qty', 'target_bought']

Distribution de la variable cible (target_bought):


target_bought
0    89.9%
1    10.1%
Name: proportion, dtype: object


Exemples de lignes:


,code_client,code_article,avg_qty,std_qty,min_qty,max_qty,total_qty,frequency,last_qty,recency_days,...,designation,is_bulk_product,nb_clients,days_since_first_order,is_new_product,client_total_products,client_total_invoices,client_avg_basket_size,target_qty,target_bought
14859,CLT104508,23129RAA4GBLUE8/256,1.0,0.000000,1.0,1.0,1.0,1,1.0,726.0,...,REDMI NOTE 13 ICE BLUE 8/256GB,False,191,881,False,114,39,4.384615,0.0,0
80178,CLT105491,23090RA98GBLA12/512,0.0,0.000000,0.0,0.0,0.0,0,0.0,9999.0,...,REDMI NOTE 13 PRO+ 5G MIDNIGHT BLACK 12/512GB,False,83,881,False,2,2,1.000000,0.0,0
25856,CLT108987,23106RN0DAWHITE4/128,5.0,1.414214,4.0,7.0,20.0,4,4.0,738.0,...,REDMI 13C GLACIER WHITE 4/128,False,190,847,False,123,91,4.736264,4.0,1
56657,CLT089765,M2349E1 BLACK,0.0,0.000000,0.0,0.0,0.0,0,0.0,9999.0,...,REDMI BUDS 6 LITE BLACK,False,132,586,False,184,64,7.906250,0.0,0
59982,CLT095704,BG6M BLACK 2025,0.0,0.000000,0.0,0.0,0.0,0,0.0,9999.0,...,TECNO SPARK GO 2025 INK BLACK 2/64,False,121,329,False,128,35,5.571429,0.0,0
